Tasks - Linear Algebra 2 (Complex)

These tasks require implementing full algorithms from scratch by combining
multiple topics from the LA-2 notebooks.
Topics: complete linear system solving, PCA whitening, Hessian computation,
scaled dot-product attention, batch normalisation, gradient flow analysis,
and a full mini transformer block.

Expected time per task: 15-25 minutes.

Task 1 - Solve via LU and Interpret Null Space

Given a nearly rank-deficient feature matrix, solve the least-squares problem,
find the null space direction, and show that perturbing the solution along the
null space leaves predictions unchanged.

In [ ]:
import numpy as np
from scipy.linalg import lu, solve_triangular

np.random.seed(0)
n = 30
x1 = np.random.randn(n)
x2 = np.random.randn(n)
# col 3 is nearly col 1 + col 2 (plus tiny noise)
x3 = x1 + x2 + 1e-6 * np.random.randn(n)
X  = np.column_stack([x1, x2, x3])
y  = 2*x1 - x2 + 0.1*np.random.randn(n)

print(f'rank of X: {np.linalg.matrix_rank(X, tol=1e-4)}  (expected 2 due to near-redundancy)')

# find least-squares solution using lstsq
w, _, _, _ = None  # np.linalg.lstsq(X, y, rcond=None)

# find null space vector via SVD
U, S, Vt = None  # np.linalg.svd(X)
print(f'singular values: {S.round(4)}')
null_vec = None  # Vt[-1]  (row of Vt with smallest singular value)

# perturb w along null space: predictions should remain unchanged
w2 = None  # w + 100.0 * null_vec

print(f'w  (original) : {w.round(4)}')
print(f'w2 (perturbed): {w2.round(4)}')
print(f'predictions same: {np.allclose(X @ w, X @ w2, atol=1e-4)}')

Task 2 - PCA Whitening from Scratch

Implement PCA whitening in 5 steps:
1. Centre the data
2. Compute covariance matrix
3. Eigendecompose (use eigh for symmetric matrices)
4. Project onto eigenvectors
5. Scale each component by 1/sqrt(eigenvalue)

Verify: the covariance of the whitened data should be the identity matrix.

In [ ]:
import numpy as np

np.random.seed(1)
cov_true = np.array([[4.0, 3.0, 1.0],
                     [3.0, 5.0, 2.0],
                     [1.0, 2.0, 3.0]])
X = np.random.multivariate_normal([2.0, 3.0, 1.0], cov_true, size=500)
n = X.shape[0]

# step 1: centre
X_c = None  # X - X.mean(axis=0)

# step 2: covariance matrix
C = None  # (1/(n-1)) * X_c.T @ X_c
print(f'covariance:\n{C.round(3)}')

# step 3: eigendecompose
eigenvalues, Q = None  # np.linalg.eigh(C)
# sort descending
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
Q = Q[:, idx]

# step 4: project
X_proj = None  # X_c @ Q

# step 5: scale by 1/sqrt(eigenvalue)
X_white = None  # X_proj / np.sqrt(eigenvalues)

C_white = (1 / (n - 1)) * X_white.T @ X_white
print(f'\nwhitened covariance (should be ~I):\n{C_white.round(3)}')
print(f'is identity: {np.allclose(C_white, np.eye(3), atol=0.05)}')

Task 3 - Hessian and Curvature Analysis

Compute the Hessian of a scalar function using PyTorch autograd.
Then use its eigenvalues to determine:
- whether the function is locally convex (all eigenvalues positive)
- the condition number of the Hessian (ratio of max to min eigenvalue)
- which direction has the steepest curvature

In [ ]:
import torch

# f(x) = 5*x0^2 + x1^2 + 0.5*x0*x1
# Hessian = [[10, 0.5], [0.5, 2]]  (constant)

def f(x):
    return 5*x[0]**2 + x[1]**2 + 0.5*x[0]*x[1]

x = torch.tensor([1.0, 2.0])

# compute Hessian
H = None  # torch.autograd.functional.hessian(f, x)

print(f'Hessian:\n{H}')
H_anal = torch.tensor([[10.0, 0.5], [0.5, 2.0]])
print(f'matches analytical: {torch.allclose(H, H_anal)}')

# eigenvalues of the Hessian (curvature in each principal direction)
eigvals = None  # torch.linalg.eigvalsh(H)
print(f'\nHessian eigenvalues: {eigvals}')
print(f'locally convex (all > 0): {bool(torch.all(eigvals > 0))}')

# condition number of the Hessian
kappa = None  # eigvals[-1] / eigvals[0]
print(f'condition number: {kappa:.4f}  (high -> slow convergence along flat direction)')

Task 4 - Scaled Dot-Product Attention from Scratch

Implement scaled dot-product attention using only NumPy.
Steps: compute scores Q @ K.T / sqrt(d_k), apply softmax, multiply by V.
Then add a causal mask so position i can only attend to positions <= i.

In [ ]:
import numpy as np

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

np.random.seed(5)
seq, d_k, d_v = 5, 8, 8
Q = np.random.randn(seq, d_k)
K = np.random.randn(seq, d_k)
V = np.random.randn(seq, d_v)

# step 1: compute raw scores Q @ K.T / sqrt(d_k)
scores = None  # your code here, shape (seq, seq)

# step 2: apply causal mask -- zero out upper triangle (future positions)
causal_mask = np.tril(np.ones((seq, seq), dtype=bool))
scores_masked = None  # np.where(causal_mask, scores, -1e9)

# step 3: softmax to get attention weights
weights = None  # softmax(scores_masked)

# step 4: weighted sum of values
output = None  # weights @ V

print(f'scores shape  : {scores.shape}')
print(f'weights shape : {weights.shape}')
print(f'output shape  : {output.shape}')
print(f'weights sum to 1: {np.allclose(weights.sum(axis=-1), 1.0)}')
print(f'\nweights (causal -- upper triangle should be ~0):\n{weights.round(3)}')

Task 5 - Batch Normalisation from Scratch

Implement batch normalisation for a linear layer output.
During training: normalise each feature using batch statistics (mean and std over the batch).
Then apply learnable gamma (scale) and beta (shift).
Also update running mean and variance for inference.

In [ ]:
import numpy as np

np.random.seed(3)
batch, features = 64, 16
X = np.random.randn(batch, features) * 3 + 5   # non-zero mean and scale

gamma = np.ones(features)    # learnable scale
beta  = np.zeros(features)   # learnable shift
eps   = 1e-5
momentum = 0.1

running_mean = np.zeros(features)
running_var  = np.ones(features)

# --- training mode ---
# compute batch mean and variance per feature
mu    = None  # X.mean(axis=0)  -- shape (features,)
var   = None  # X.var(axis=0)   -- shape (features,)

# update running statistics
running_mean = (1 - momentum) * running_mean + momentum * mu
running_var  = (1 - momentum) * running_var  + momentum * var

# normalise
X_norm = None  # (X - mu) / np.sqrt(var + eps)

# scale and shift
X_bn = None  # gamma * X_norm + beta

print(f'input mean (first 4 features) : {X[:, :4].mean(axis=0).round(4)}')
print(f'output mean (first 4 features): {X_bn[:, :4].mean(axis=0).round(6)}  (should be ~0)')
print(f'output std  (first 4 features): {X_bn[:, :4].std(axis=0).round(4)}   (should be ~1)')

# --- inference mode: use running statistics ---
X_test = np.random.randn(8, features) * 3 + 5
X_test_norm = (X_test - running_mean) / np.sqrt(running_var + eps)
X_test_bn   = None  # gamma * X_test_norm + beta
print(f'\ninference output std: {X_test_bn.std(axis=0)[:4].round(4)}')

Task 6 - Gradient Flow Analysis

Simulate how gradient magnitude changes as it propagates backwards through multiple layers.
Compare three initialisation strategies: naive (std=1), too small (std=0.01), and He init.
Show which strategy keeps gradients stable across 20 layers.

In [ ]:
import numpy as np

np.random.seed(0)
num_layers, d = 20, 64

def simulate_backward(num_layers, d, init_std, activation='relu'):
    grad_norms = []
    g = np.ones(d)   # start with unit gradient
    for _ in range(num_layers):
        W = np.random.randn(d, d) * init_std
        # simplified backward: gradient through W is W.T @ g
        # for ReLU: approximately half the neurons are active
        g = W.T @ g
        if activation == 'relu':
            mask = np.random.rand(d) > 0.5   # ~50% neurons active
            g = g * mask
        grad_norms.append(np.linalg.norm(g))
    return grad_norms

# He init std for ReLU: sqrt(2/d)
he_std = None  # np.sqrt(2.0 / d)

norms_naive = simulate_backward(num_layers, d, init_std=1.0)
norms_small = simulate_backward(num_layers, d, init_std=0.01)
norms_he    = None  # simulate_backward(num_layers, d, init_std=he_std)

print('gradient norms across 20 layers (sampled at layers 1, 5, 10, 15, 20):')
for layer in [0, 4, 9, 14, 19]:
    print(f'  layer {layer+1:2d}  naive={norms_naive[layer]:.2e}  '
          f'small={norms_small[layer]:.2e}  '
          f'he={norms_he[layer]:.4f}')

Task 7 - Multi-Head Attention from Scratch

Implement multi-head attention using NumPy.
Split d_model into h heads of size d_k = d_model // h.
Run scaled dot-product attention for each head in parallel, concatenate, then project.

In [ ]:
import numpy as np

def softmax_stable(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

np.random.seed(42)
seq, d_model, h = 6, 16, 4
d_k = d_model // h   # = 4

# weight matrices (randomly initialised)
s = np.sqrt(2.0 / d_model)
WQ = np.random.randn(d_model, d_model) * s
WK = np.random.randn(d_model, d_model) * s
WV = np.random.randn(d_model, d_model) * s
WO = np.random.randn(d_model, d_model) * s

X = np.random.randn(seq, d_model)   # input embeddings

# project to Q, K, V
Q_all = None  # X @ WQ  -- shape (seq, d_model)
K_all = None  # X @ WK
V_all = None  # X @ WV

# run each head
head_outputs = []
for i in range(h):
    Q_i = Q_all[:, i*d_k:(i+1)*d_k]  # (seq, d_k)
    K_i = None  # your code here
    V_i = None  # your code here

    scores_i  = None  # Q_i @ K_i.T / np.sqrt(d_k)
    weights_i = None  # softmax_stable(scores_i)
    head_i    = None  # weights_i @ V_i
    head_outputs.append(head_i)

# concatenate heads and apply output projection
concat = None  # np.concatenate(head_outputs, axis=-1)  -- (seq, d_model)
output = None  # concat @ WO                            -- (seq, d_model)

print(f'input shape : {X.shape}')
print(f'output shape: {output.shape}  (should match input shape)')

Task 8 - Full Transformer Block

Combine everything into one complete transformer block:
1. Multi-head self-attention (from task 7)
2. Residual connection + layer normalisation
3. Feed-forward network: Linear -> ReLU -> Linear (He init)
4. Residual connection + layer normalisation

Verify output shape matches input shape and that layer norm stabilises each token.

In [ ]:
import numpy as np

def softmax_stable(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def layer_norm(x, eps=1e-5):
    mu  = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1,  keepdims=True)
    return (x - mu) / (std + eps)

def mha(X, WQ, WK, WV, WO, h):
    seq, d = X.shape
    dk = d // h
    Q, K, V = X @ WQ, X @ WK, X @ WV
    heads = []
    for i in range(h):
        Qi = Q[:, i*dk:(i+1)*dk]
        Ki = K[:, i*dk:(i+1)*dk]
        Vi = V[:, i*dk:(i+1)*dk]
        w = softmax_stable(Qi @ Ki.T / dk**0.5)
        heads.append(w @ Vi)
    return np.concatenate(heads, axis=-1) @ WO

np.random.seed(7)
seq, d_model, h, d_ff = 8, 32, 4, 64
s    = np.sqrt(2.0 / d_model)
s_ff = np.sqrt(2.0 / d_model)   # He init for ReLU in FFN

WQ = np.random.randn(d_model, d_model) * s
WK = np.random.randn(d_model, d_model) * s
WV = np.random.randn(d_model, d_model) * s
WO = np.random.randn(d_model, d_model) * s
W1 = np.random.randn(d_model, d_ff) * s_ff
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d_model) * s
b2 = np.zeros(d_model)

X = np.random.randn(seq, d_model)

# --- sublayer 1: multi-head attention + residual + LN ---
attn_out = None  # mha(X, WQ, WK, WV, WO, h)
X2 = None        # layer_norm(X + attn_out)       -- residual connection

# --- sublayer 2: feed-forward + residual + LN ---
ff_hidden = None  # np.maximum(0, X2 @ W1 + b1)   -- ReLU
ff_out    = None  # ff_hidden @ W2 + b2
X3 = None         # layer_norm(X2 + ff_out)        -- residual connection

print(f'input  shape: {X.shape}')
print(f'output shape: {X3.shape}  (should match)')
print(f'\noutput per-token mean: {X3.mean(axis=-1).round(6)}  (LN -> ~0)')
print(f'output per-token std : {X3.std(axis=-1).round(4)}   (LN -> ~1)')